In [3]:
import os
import re
import glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG (YOUR PATHS)
# =========================
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"

COMMUNE_NAME_COL = "nom"       # adjust if needed (e.g., NOM)
COMMUNE_CODE_COL = "insee_com" # adjust if needed (e.g., INSEE_COM)

OUT_DIR = os.path.join(FLOODS_BASE, "_outputs_inseecom")
OUT_WIDE = os.path.join(OUT_DIR, "events_communes_wide_all_years.csv")
OUT_LONG = os.path.join(OUT_DIR, "events_communes_long_all_years.csv")

# README: depths are in cm, dtype uint16, dummy 9999 = water bodies
THRESHOLD_CM = 0.0
IGNORE_VALUES = {9999}
ALL_TOUCHED = True

ERROR_SAMPLE_LIMIT = 50

# =========================
# HELPERS
# =========================
def find_flood_roots(base_dir: str | Path, suffix: str = "_filtered") -> list[str]:
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(f"FLOODS_BASE not found: {base_dir}")
    roots = sorted(str(p) for p in base_dir.iterdir() if p.is_dir() and p.name.endswith(suffix))
    if not roots:
        raise FileNotFoundError(f"No '*{suffix}' folders found under: {base_dir}")
    return roots

def collect_tifs(roots: list[str]) -> list[str]:
    tifs = []
    for root in roots:
        tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
        tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))
    return sorted(set(tifs))

def infer_year_from_path(fp: str) -> int | None:
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    base = os.path.basename(fp)

    m = re.search(
        r"WD_MERGE_"
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2})"
        r"_duration_(?P<duration>\d+)_days"
        r"_cluster_(?P<cluster>\d+)"
        r"_A0_",
        base,
        flags=re.IGNORECASE
    )
    if not m:
        return None

    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": m.group("start"),
        "end_date": m.group("end"),
        "duration_days": int(m.group("duration")),
        "cluster_id": int(m.group("cluster")),
    }

def event_id_from_meta(meta: dict) -> str:
    return f"{meta['start_date']}---{meta['end_date']}_cluster_{meta['cluster_id']}"

def normalize_insee_code(series: pd.Series) -> pd.Series:
    """
    Robust normalization:
    - keeps strings
    - removes accidental '.0' float suffix
    - strips spaces
    - zero-pads to 5 digits
    """
    s = series.astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    # If some values are already alphanumeric (e.g., special cases), zfill still safe for short strings
    s = s.str.zfill(5)
    return s

def commune_is_affected(src, geom, threshold_cm: float):
    """
    Returns (affected: bool, max_cm: float|None)
    NOTE: geom MUST be in src.crs
    """
    data, _ = mask(
        src,
        [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )

    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()

    if vals.size == 0:
        return False, None

    if IGNORE_VALUES:
        # vectorized filter for all ignored values
        mask_keep = np.ones(vals.shape, dtype=bool)
        for v in IGNORE_VALUES:
            mask_keep &= (vals != v)
        vals = vals[mask_keep]
        if vals.size == 0:
            return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# =========================
# LOAD COMMUNES (AdminExpress)
# =========================
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
communes = communes[communes.geometry.notna() & ~communes.geometry.is_empty].copy()

# Validate required columns
required_cols = [COMMUNE_NAME_COL, COMMUNE_CODE_COL, "geometry"]
missing_cols = [c for c in required_cols if c not in communes.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in communes layer '{COMMUNES_LAYER}': {missing_cols}")

# Normalize INSEE code (you said it's always there, but normalize format anyway)
communes[COMMUNE_CODE_COL] = normalize_insee_code(communes[COMMUNE_CODE_COL])

# Optional: normalize names to string
communes[COMMUNE_NAME_COL] = communes[COMMUNE_NAME_COL].astype(str).str.strip()

# Keep only necessary columns for speed/memory in spatial workflow
communes = communes[[COMMUNE_CODE_COL, COMMUNE_NAME_COL, "geometry"]].copy()

# Candidate filtering in EPSG:4326
communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex

# =========================
# DISCOVER FLOODS + GROUP FILES BY EVENT
# =========================
roots = find_flood_roots(FLOODS_BASE, suffix="_filtered")
print("Detected flood folders:")
for r in roots:
    print(" -", r)

tifs = collect_tifs(roots)
print("\nTotal TIFs found:", len(tifs))

event_to_files = defaultdict(list)
event_meta = {}  # event_id -> representative meta (first seen tile)
unparsed_files = []

for fp in tifs:
    meta = parse_event_from_name(fp)
    if meta is None:
        unparsed_files.append(fp)
        continue

    eid = event_id_from_meta(meta)
    event_to_files[eid].append(fp)

    # store one representative meta per event (first file)
    if eid not in event_meta:
        m = meta.copy()
        m["event_id"] = eid
        event_meta[eid] = m

# Sort tile list per event for deterministic processing
for eid in event_to_files:
    event_to_files[eid] = sorted(set(event_to_files[eid]))

print("Unparsed files:", len(unparsed_files))
print("Distinct events:", len(event_to_files))

# =========================
# MAIN LOOP (per EVENT, multiple tiles)
# =========================
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

wide_rows = []
long_rows = []

diag = Counter()
error_counts = Counter()
error_samples = []

for eid, files in tqdm(event_to_files.items(), desc="Processing events"):
    diag["events_total"] += 1
    meta = event_meta[eid]

    # Track max depth per commune across all tiles for this event
    # key = insee code (string)
    # value = {"max_cm": float, "commune_code": str, "commune": str}
    commune_max = {}

    tested_total = 0
    overlap_skips_total = 0
    any_candidates = False
    tiles_processed = 0

    for fp in files:
        try:
            with rasterio.open(fp) as src:
                if src.crs is None:
                    diag["tiles_no_crs"] += 1
                    continue

                tiles_processed += 1

                # ---- STEP 1: bbox filtering in WGS84 ----
                try:
                    b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
                except Exception as e:
                    msg = f"transform_bounds_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue

                raster_bbox_wgs84 = box(*b)

                cand_idx = list(sindex.intersection(raster_bbox_wgs84.bounds))
                if not cand_idx:
                    continue

                cands = communes_wgs84.iloc[cand_idx]
                cands = cands[cands.intersects(raster_bbox_wgs84)]
                if cands.empty:
                    continue

                any_candidates = True

                # ---- STEP 2: reproject candidates to raster CRS for mask() ----
                cands_src = cands.to_crs(src.crs).copy()

                # Shapely 2 make_valid if available
                try:
                    cands_src["geometry"] = cands_src.geometry.make_valid()
                except Exception:
                    pass

                # Drop invalid/empty geoms after make_valid
                cands_src = cands_src[cands_src.geometry.notna() & ~cands_src.geometry.is_empty]
                if cands_src.empty:
                    continue

                raster_box_src = box(*src.bounds)

                for row_idx, crow in cands_src.iterrows():
                    geom = crow.geometry
                    if geom is None or geom.is_empty:
                        continue

                    if not geom.intersects(raster_box_src):
                        overlap_skips_total += 1
                        continue

                    tested_total += 1

                    try:
                        ok, max_cm = commune_is_affected(src, geom, THRESHOLD_CM)
                    except Exception as e:
                        msg = f"commune_mask_error: {repr(e)}"
                        error_counts[msg] += 1
                        if len(error_samples) < ERROR_SAMPLE_LIMIT:
                            error_samples.append((fp, msg))
                        continue

                    if not ok or max_cm is None:
                        continue

                    # You said insee_com is always present -> use it as stable key
                    ccode = str(crow[COMMUNE_CODE_COL]).strip()
                    ccode = re.sub(r"\.0$", "", ccode)
                    ccode = ccode.zfill(5)

                    cname = str(crow[COMMUNE_NAME_COL]).strip() if pd.notna(crow[COMMUNE_NAME_COL]) else None

                    prev = commune_max.get(ccode)
                    if (prev is None) or (max_cm > prev["max_cm"]):
                        commune_max[ccode] = {
                            "max_cm": float(max_cm),
                            "commune_code": ccode,
                            "commune": cname,
                        }

        except Exception as e:
            msg = f"tile_error: {repr(e)}"
            error_counts[msg] += 1
            if len(error_samples) < ERROR_SAMPLE_LIMIT:
                error_samples.append((fp, msg))
            continue

    if any_candidates:
        diag["events_with_candidates_bbox"] += 1

    if commune_max:
        diag["events_with_affected"] += 1

    # Build long rows (one row per affected commune for this event)
    affected_names = []
    for ccode, rec in commune_max.items():
        long_rows.append({
            "event_id": meta["event_id"],
            "year": meta["year"],
            "start_date": meta["start_date"],
            "end_date": meta["end_date"],
            "duration_days": meta["duration_days"],
            "cluster_id": meta["cluster_id"],
            "commune_code": rec["commune_code"],
            "commune": rec["commune"],
            "max_cm": rec["max_cm"],
        })
        if rec["commune"]:
            affected_names.append(rec["commune"])

    # Wide row (one row per event)
    wide_rows.append({
        "event_id": meta["event_id"],
        "year": meta["year"],
        "start_date": meta["start_date"],
        "end_date": meta["end_date"],
        "duration_days": meta["duration_days"],
        "cluster_id": meta["cluster_id"],
        "n_tiles": len(files),                     # all grouped files
        "n_tiles_processed": tiles_processed,      # tiles successfully opened with CRS
        "event_path": meta["event_path"],          # representative (first seen tile)
        "n_communes": len(commune_max),
        "communes": "; ".join(sorted(set(affected_names))),
        "n_candidates_tested": tested_total,
        "n_overlap_skipped": overlap_skips_total,
    })

# =========================
# BUILD DATAFRAMES + SAVE
# =========================
wide_cols = [
    "event_id", "year", "start_date", "end_date", "duration_days", "cluster_id",
    "n_tiles", "n_tiles_processed", "event_path",
    "n_communes", "communes",
    "n_candidates_tested", "n_overlap_skipped",
]
long_cols = [
    "event_id", "year", "start_date", "end_date", "duration_days", "cluster_id",
    "commune_code", "commune", "max_cm",
]

df_wide = pd.DataFrame(wide_rows)
df_long = pd.DataFrame(long_rows)

# Ensure columns exist even if empty
for c in wide_cols:
    if c not in df_wide.columns:
        df_wide[c] = pd.Series(dtype="object")
df_wide = df_wide[wide_cols]

for c in long_cols:
    if c not in df_long.columns:
        df_long[c] = pd.Series(dtype="object")
df_long = df_long[long_cols]

# Sorting
if not df_wide.empty:
    df_wide = df_wide.sort_values(
        ["year", "start_date", "end_date", "cluster_id"],
        na_position="last"
    ).reset_index(drop=True)

if not df_long.empty:
    df_long = df_long.sort_values(
        ["year", "start_date", "end_date", "cluster_id", "commune_code"],
        na_position="last"
    ).reset_index(drop=True)

# Save outputs
df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved combined outputs:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

print("\nDiagnostics:")
for k in ["events_total", "events_with_candidates_bbox", "events_with_affected", "tiles_no_crs"]:
    print(f" - {k}: {diag.get(k, 0)}")

print("\nTop error types:")
for msg, n in error_counts.most_common(10):
    print(f" - {n}  {msg}")

if error_samples:
    print("\nError samples:")
    for fp, msg in error_samples[:10]:
        print(" -", os.path.basename(fp), "->", msg)

if unparsed_files:
    print("\nSample unparsed filenames:")
    for fp in unparsed_files[:10]:
        print(" -", os.path.basename(fp))

Detected flood folders:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2015_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2016_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2017_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2018_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2019_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2020_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2022_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered

Total TIFs found: 1880
Unparsed files: 0
Distinct events: 1746


Processing events: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 1746/1746 [2:26:46<00:00,  5.04s/it]


Saved combined outputs:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_inseecom\events_communes_wide_all_years.csv rows: 1746
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs_inseecom\events_communes_long_all_years.csv rows: 5378

Diagnostics:
 - events_total: 1746
 - events_with_candidates_bbox: 468
 - events_with_affected: 135
 - tiles_no_crs: 0

Top error types:
